# MEGA to Google Drive Cloud GUI Transfer
This tool runs completely in the cloud on Google's high-speed servers to transfer files from MEGA directly to your Google Drive, bypassing local IP limits.

### Step 1: Connect your Google Drive
Run this cell below, click the authorization link, and sign in. (Alternatively, click the **Files 📁** icon in the left sidebar and click the **Mount Drive** folder button).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Install Transfer Engine
Run this cell to set up the official MEGA downloader engine. This only needs to be run once per session.

In [ ]:
!wget -q https://mega.nz/linux/repo/xUbuntu_22.04/amd64/megacmd-xUbuntu_22.04_amd64.deb
!sudo apt-get install "$PWD/megacmd-xUbuntu_22.04_amd64.deb" -y

### Step 3: Enter Details & Transfer
Use the form fields on the right to enter your public MEGA link and your destination folder name in Google Drive, then click **Play (Run)** to start streaming.

In [ ]:
#@title Transfer Configuration Form { display-mode: "form" }
MEGA_LINK = "" #@param {type:"string"}
GDRIE_FOLDER_NAME = "MEGA_Transfers" #@param {type:"string"}

import os
import subprocess
import time
import sys
import fcntl
import pty

TARGET_DIR = f"/content/drive/MyDrive/{GDRIE_FOLDER_NAME}"
os.makedirs(TARGET_DIR, exist_ok=True)

if not MEGA_LINK or MEGA_LINK == "https://mega.nz/folder/...":
    print("\033[91m[ERROR] Please paste your public MEGA link into the form field above!\033[0m")
else:
    print(f"\033[94m[INFO] Destination Folder: Google Drive -> {GDRIE_FOLDER_NAME}\033[0m")
    print(f"\033[94m[INFO] Starting stream transfer... Please wait.\033[0m")
    print(f"\033[1m[NOTE] Once the progress bar hits 100%, Google Drive will take 1-2 minutes to write the file. The script will auto-terminate when complete!\033[0m\n")
    
    # Clean shutdown of any previous sessions to start fresh
    subprocess.run(["mega-quit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(1)
    
    # Open a Pseudo-Terminal (PTY) master/slave pair to force interactive outputs (single-line carriage return updates)
    master, slave = pty.openpty()
    
    # Launch mega-get as a subprocess tied to the PTY
    cmd = ["mega-get", MEGA_LINK, TARGET_DIR]
    process = subprocess.Popen(cmd, stdout=slave, stderr=slave, text=True, close_fds=True)
    
    # Close slave file descriptor in parent since child has inherited it
    os.close(slave)
    
    # Set PTY master to non-blocking mode
    fl = fcntl.fcntl(master, fcntl.F_GETFL)
    fcntl.fcntl(master, fcntl.F_SETFL, fl | os.O_NONBLOCK)
    
    last_transfers_check = time.time()
    
    while True:
        # Try to read stream chunks from the master PTY
        try:
            chunk = os.read(master, 1024)
            if chunk:
                sys.stdout.write(chunk.decode('utf-8', errors='ignore'))
                sys.stdout.flush()
        except OSError:
            pass  # No new characters in buffer
            
        # Check if process exited naturally
        if process.poll() is not None:
            # Clean up remaining buffer output
            try:
                remaining = os.read(master, 4096)
                if remaining:
                    sys.stdout.write(remaining.decode('utf-8', errors='ignore'))
                    sys.stdout.flush()
            except:
                pass
            break
            
        # Check mega-transfers daemon status every 5 seconds
        now = time.time()
        if now - last_transfers_check > 5.0:
            last_transfers_check = now
            try:
                res = subprocess.run(["mega-transfers"], capture_output=True, text=True)
                # If daemon reports empty queue, we are done!
                if "No active transfers" in res.stdout or "No active transfers" in res.stderr:
                    print("\n\033[94m[INFO] Cloud transfer verified complete. Finalizing...\033[0m")
                    break
            except:
                pass
                
        time.sleep(0.1)
        
    # Clean up PTY file descriptor
    try:
        os.close(master)
    except:
        pass
        
    # Force terminate mega-get client if still hung
    if process.poll() is None:
        process.terminate()
        process.wait()
        
    # Stop the background daemon to release system resources
    subprocess.run(["mega-quit"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print("\033[92m[SUCCESS] Transfer queue completed!\033[0m")

### Step 3.5: Monitor Active Transfers (Optional)
If you want to check the progress, transfer speed, and queue status of your ongoing downloads, run this cell at any time during a transfer.

In [ ]:
!mega-transfers

### Step 4: Force Sync & Disconnect (Optional)
Run this cell once all your transfers are complete to instantly flush all cached data to Google Drive servers (so they appear on your phone/browser immediately) and safely unmount the connection.

In [ ]:
from google.colab import drive
print("[INFO] Flushing cache and disconnecting Drive...")
drive.flush_and_unmount()
print("[SUCCESS] Google Drive successfully synced and unmounted!")